# ADMET Hamiltonian Curation Notebook

This notebook handles ADMET dataset loading, molecular fingerprint generation, feature selection, feature-interaction analysis, and construction of Hamiltonian payloads for downstream LCU/block-encoding workflows.

The notebook stops at Hamiltonian curation and export. QBE generation, LCU validation, block encoding, reflection, and walk-operator logic are handled separately in the class-based LCU framework.

In [1]:
import sys
print(sys.version)

3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 21.0.0 (clang-2100.0.123.102)]


Initialization 

In [2]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem

from tdc.single_pred import ADME

print("Environment ready.")

Environment ready.


ADMET Dataset Download 

In [3]:
from tdc.single_pred import ADME, Tox

admet_datasets = [
    ("ADME", "CYP3A4_Substrate_CarbonMangels"),
    ("ADME", "CYP2D6_Substrate_CarbonMangels"),
    ("ADME", "CYP2C9_Substrate_CarbonMangels"),
    ("Tox", "hERG"),
    ("Tox", "AMES"),
    ("ADME", "BBB_Martins"),
    ("ADME", "PGP_Broccatelli"),
    ("ADME", "Bioavailability_Ma"),
    ("Tox", "DILI"),
    ("ADME", "HIA_Hou"),
]

datasets = {}

for task_type, name in admet_datasets:
    print(f"Downloading {name}...")
    
    if task_type == "ADME":
        data = ADME(name=name)
    elif task_type == "Tox":
        data = Tox(name=name)
    
    df = data.get_data()
    datasets[name] = df
    
    print(name, df.shape, df.columns.tolist())
    display(df.head())

Found local copy...
Loading...
Done!


CYP3A4_Substrate_CarbonMangels (670, 3) ['Drug_ID', 'Drug', 'Y']


,Drug_ID,Drug,Y
0,abacavir,Nc1nc(NC2CC2)c2ncn([C@H]3C=C[C@@H](CO)C3)c2n1,0
1,abecarnil,COCc1c(C(=O)OC(C)C)ncc2[nH]c3ccc(OCc4ccccc4)cc...,0
2,abiraterone,C[C@]12CC[C@H]3[C@@H](CC=C4C[C@@H](O)CC[C@@]43...,0
3,acebutolol,CCCC(=O)Nc1ccc(OC[C@@H](O)CNC(C)C)c(C(C)=O)c1,0
4,aceclofenac,O=C(O)COC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl,0


Found local copy...
Loading...
Done!


CYP2D6_Substrate_CarbonMangels (667, 3) ['Drug_ID', 'Drug', 'Y']


,Drug_ID,Drug,Y
0,abacavir,Nc1nc(NC2CC2)c2ncn([C@H]3C=C[C@@H](CO)C3)c2n1,0
1,abecarnil,COCc1c(C(=O)OC(C)C)ncc2[nH]c3ccc(OCc4ccccc4)cc...,0
2,abiraterone,C[C@]12CC[C@H]3[C@@H](CC=C4C[C@@H](O)CC[C@@]43...,0
3,acebutolol,CCCC(=O)Nc1ccc(OC[C@@H](O)CNC(C)C)c(C(C)=O)c1,0
4,aceclofenac,O=C(O)COC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl,0


... (20 outputs left out)

,Drug_ID,Drug,Y
0,187.0,CC(=O)OCC[N+](C)(C)C,0.0
1,247.0,C[N+](C)(C)CC(=O)[O-],0.0
2,298.0,O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl,0.0
3,338.0,O=C(O)c1ccccc1O,0.0
4,444.0,CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1,0.0


Found local copy...
Loading...
Done!


HIA_Hou (578, 3) ['Drug_ID', 'Drug', 'Y']


,Drug_ID,Drug,Y
0,Acetanilide,CC(=O)Nc1ccccc1,1
1,Acetazolamide.mol,CC(=O)Nc1nnc(S(N)(=O)=O)s1,1
2,Alfacalcidol,C=C1/C(=C\C=C2/CCC[C@]3(C)[C@@H]2CC[C@H]3[C@@H...,1
3,Aminopyrine,Cc1c(N(C)C)c(=N)n(-c2ccccc2)n1C,1
4,Amosulalol.mol,COc1ccccc1OCCNC[C@@H](O)c1ccc(C)cc1S(N)(=O)=O,1


Molecular Fingerprint Generation

In [17]:
# Morgan count fingerprints for all 10 datasets from PolarisQB
# Maurice-compatible: count-valued features, not binary bit vectors

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd


def smiles_to_morgan_count_fp(smiles, n_bits=1024, radius=2):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fp = AllChem.GetHashedMorganFingerprint(
        mol,
        radius,
        nBits=n_bits
    )

    arr = np.zeros((n_bits,), dtype=float)

    for idx, count in fp.GetNonzeroElements().items():
        arr[idx] = float(count)

    return arr


fingerprint_data = {}

for name, df in datasets.items():
    print(f"Processing {name}...")

    smiles_col = "Drug"
    label_col = "Y"

    fps = []
    labels = []
    valid_smiles = []

    for _, row in df.iterrows():
        fp = smiles_to_morgan_count_fp(row[smiles_col])

        if fp is not None:
            fps.append(fp)
            labels.append(row[label_col])
            valid_smiles.append(row[smiles_col])

    X = np.array(fps, dtype=float)
    y = np.array(labels)

    fingerprint_data[name] = {
        "X": X,
        "y": y,
        "smiles": valid_smiles,
        "fingerprint_type": "Morgan count fingerprint",
        "n_bits": 1024,
        "radius": 2,
    }

    print(f"{name}: X shape = {X.shape}, y shape = {y.shape}")
    print(f"{name}: min={X.min()}, max={X.max()}, mean={X.mean():.6f}")
    print("-" * 60)

Processing CYP3A4_Substrate_CarbonMangels...
CYP3A4_Substrate_CarbonMangels: X shape = (670, 1024), y shape = (670,)
CYP3A4_Substrate_CarbonMangels: min=0.0, max=25.0, mean=0.066768
------------------------------------------------------------
Processing CYP2D6_Substrate_CarbonMangels...
CYP2D6_Substrate_CarbonMangels: X shape = (667, 1024), y shape = (667,)
CYP2D6_Substrate_CarbonMangels: min=0.0, max=25.0, mean=0.066185
------------------------------------------------------------
Processing CYP2C9_Substrate_CarbonMangels...
CYP2C9_Substrate_CarbonMangels: X shape = (669, 1024), y shape = (669,)
CYP2C9_Substrate_CarbonMangels: min=0.0, max=25.0, mean=0.066352
------------------------------------------------------------
Processing hERG...
hERG: X shape = (655, 1024), y shape = (655,)
hERG: min=0.0, max=36.0, mean=0.073156
------------------------------------------------------------
Processing AMES...


[16:06:17] WARNING: not removing hydrogen atom without neighbors
[16:06:17] WARNING: not removing hydrogen atom without neighbors


AMES: X shape = (7278, 1024), y shape = (7278,)
AMES: min=0.0, max=35.0, mean=0.045432
------------------------------------------------------------
Processing BBB_Martins...


[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not removing hydrogen atom without neighbors
[16:06:18] WARNING: not r

BBB_Martins: X shape = (2030, 1024), y shape = (2030,)
BBB_Martins: min=0.0, max=34.0, mean=0.065515
------------------------------------------------------------
Processing PGP_Broccatelli...
PGP_Broccatelli: X shape = (1218, 1024), y shape = (1218,)
PGP_Broccatelli: min=0.0, max=21.0, mean=0.075204
------------------------------------------------------------
Processing Bioavailability_Ma...
Bioavailability_Ma: X shape = (640, 1024), y shape = (640,)
Bioavailability_Ma: min=0.0, max=24.0, mean=0.066095
------------------------------------------------------------
Processing DILI...
DILI: X shape = (475, 1024), y shape = (475,)
DILI: min=0.0, max=21.0, mean=0.064398
------------------------------------------------------------
Processing HIA_Hou...
HIA_Hou: X shape = (578, 1024), y shape = (578,)
HIA_Hou: min=0.0, max=24.0, mean=0.062159
------------------------------------------------------------


In [18]:
#test 
name = "BBB_Martins"

print("Unique values sample:")
print(np.unique(fingerprint_data[name]["X"])[:20])

print("Max count:", fingerprint_data[name]["X"].max())

Unique values sample:
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19.]
Max count: 34.0


Feature Selection

In [19]:
from sklearn.feature_selection import mutual_info_classif
import numpy as np
import pandas as pd

top_k = 100

selected_feature_data = {}
mi_summary_rows = []

for name, data in fingerprint_data.items():
    print(f"Processing MI feature selection for {name}...")
    
    X = data["X"].astype(float)
    y = data["y"]
    
    mi_scores = mutual_info_classif(
        X,
        y,
        discrete_features=False,
        random_state=42
    )
    
    feature_ranking = np.argsort(mi_scores)[::-1]
    selected_features = feature_ranking[:top_k]
    
    X_selected = X[:, selected_features]
    
    # MinMax scale selected count features to [0, 1]
    X_min = X_selected.min(axis=0)
    X_max = X_selected.max(axis=0)
    denom = np.where((X_max - X_min) == 0, 1.0, X_max - X_min)
    X_selected_scaled = (X_selected - X_min) / denom
    
    selected_feature_data[name] = {
        "X_selected": X_selected,
        "X_selected_scaled": X_selected_scaled,
        "y": y,
        "selected_features": selected_features,
        "mi_scores": mi_scores,
        "selected_mi_scores": mi_scores[selected_features],
        "smiles": data["smiles"],
        "fingerprint_type": data.get("fingerprint_type", "Morgan count fingerprint"),
        "scaling": {
            "method": "minmax",
            "feature_min": X_min,
            "feature_max": X_max,
        },
    }
    
    mi_summary_rows.append({
        "dataset": name,
        "n_samples": X.shape[0],
        "n_original_features": X.shape[1],
        "n_selected_features": X_selected.shape[1],
        "top_feature": int(selected_features[0]),
        "top_mi_score": float(mi_scores[selected_features[0]]),
        "selected_scaled_min": float(X_selected_scaled.min()),
        "selected_scaled_max": float(X_selected_scaled.max()),
    })
    
    print(f"{name}: selected X shape = {X_selected.shape}")
    print(f"{name}: scaled X shape   = {X_selected_scaled.shape}")
    print(f"{name}: scaled min={X_selected_scaled.min():.6f}, max={X_selected_scaled.max():.6f}")
    print(f"Top feature: {selected_features[0]}, MI = {mi_scores[selected_features[0]]:.6f}")
    print("-" * 60)

mi_summary = pd.DataFrame(mi_summary_rows)
mi_summary

Processing MI feature selection for CYP3A4_Substrate_CarbonMangels...
CYP3A4_Substrate_CarbonMangels: selected X shape = (670, 100)
CYP3A4_Substrate_CarbonMangels: scaled X shape   = (670, 100)
CYP3A4_Substrate_CarbonMangels: scaled min=0.000000, max=1.000000
Top feature: 472, MI = 0.069862
------------------------------------------------------------
Processing MI feature selection for CYP2D6_Substrate_CarbonMangels...
CYP2D6_Substrate_CarbonMangels: selected X shape = (667, 100)
CYP2D6_Substrate_CarbonMangels: scaled X shape   = (667, 100)
CYP2D6_Substrate_CarbonMangels: scaled min=0.000000, max=1.000000
Top feature: 145, MI = 0.053745
------------------------------------------------------------
Processing MI feature selection for CYP2C9_Substrate_CarbonMangels...
CYP2C9_Substrate_CarbonMangels: selected X shape = (669, 100)
CYP2C9_Substrate_CarbonMangels: scaled X shape   = (669, 100)
CYP2C9_Substrate_CarbonMangels: scaled min=0.000000, max=1.000000
Top feature: 379, MI = 0.045876
--

,dataset,n_samples,n_original_features,n_selected_features,top_feature,top_mi_score,selected_scaled_min,selected_scaled_max
0,CYP3A4_Substrate_CarbonMangels,670,1024,100,472,0.069862,0.0,1.0
1,CYP2D6_Substrate_CarbonMangels,667,1024,100,145,0.053745,0.0,1.0
2,CYP2C9_Substrate_CarbonMangels,669,1024,100,379,0.045876,0.0,1.0
3,hERG,655,1024,100,849,0.111340,0.0,1.0
4,AMES,7278,1024,100,753,0.038495,0.0,1.0
5,BBB_Martins,2030,1024,100,807,0.107263,0.0,1.0
6,PGP_Broccatelli,1218,1024,100,849,0.208640,0.0,1.0
7,Bioavailability_Ma,640,1024,100,662,0.047389,0.0,1.0
8,DILI,475,1024,100,356,0.100729,0.0,1.0
9,HIA_Hou,578,1024,100,807,0.096099,0.0,1.0


In [20]:
# test 
name = "BBB_Martins"

print("Raw selected sample:", selected_feature_data[name]["X_selected"][0, :10])
print("Scaled selected sample:", np.round(selected_feature_data[name]["X_selected_scaled"][0, :10], 6))
print("Selected feature indices:", selected_feature_data[name]["selected_features"][:10])

Raw selected sample: [1. 0. 0. 0. 0. 3. 0. 0. 0. 0.]
Scaled selected sample: [0.055556 0.       0.       0.       0.       0.1      0.       0.
 0.       0.      ]
Selected feature indices: [ 807 1019    5  650  233  356  573  578  611  387]


Feature-Feature Interaction

In [21]:
import numpy as np
import pandas as pd

feature_interaction_data = {}

for name, data in selected_feature_data.items():
    print(f"Computing feature interactions for {name}...")
    
    X_selected = data["X_selected_scaled"]
    
    corr_matrix = np.corrcoef(X_selected, rowvar=False)
    corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
    
    interaction_matrix = np.abs(corr_matrix)
    np.fill_diagonal(interaction_matrix, 0.0)
    
    feature_interaction_data[name] = {
        "interaction_matrix": interaction_matrix,
        "corr_matrix": corr_matrix,
        "selected_features": data["selected_features"],
        "X_selected": data["X_selected"],
        "X_selected_scaled": data["X_selected_scaled"],
        "y": data["y"],
        "smiles": data["smiles"],
    }
    
    print(f"{name}: interaction matrix shape = {interaction_matrix.shape}")

Computing feature interactions for CYP3A4_Substrate_CarbonMangels...
CYP3A4_Substrate_CarbonMangels: interaction matrix shape = (100, 100)
Computing feature interactions for CYP2D6_Substrate_CarbonMangels...
CYP2D6_Substrate_CarbonMangels: interaction matrix shape = (100, 100)
Computing feature interactions for CYP2C9_Substrate_CarbonMangels...
CYP2C9_Substrate_CarbonMangels: interaction matrix shape = (100, 100)
Computing feature interactions for hERG...
hERG: interaction matrix shape = (100, 100)
Computing feature interactions for AMES...
AMES: interaction matrix shape = (100, 100)
Computing feature interactions for BBB_Martins...
BBB_Martins: interaction matrix shape = (100, 100)
Computing feature interactions for PGP_Broccatelli...
PGP_Broccatelli: interaction matrix shape = (100, 100)
Computing feature interactions for Bioavailability_Ma...
Bioavailability_Ma: interaction matrix shape = (100, 100)
Computing feature interactions for DILI...
DILI: interaction matrix shape = (100, 10

Check

In [22]:
name = "BBB_Martins"

interaction_matrix = feature_interaction_data[name]["interaction_matrix"]

print("Max interaction:", interaction_matrix.max())
print("Mean interaction:", interaction_matrix.mean())
print("Non-zero entries:", np.count_nonzero(interaction_matrix))

Max interaction: 0.7673597376145127
Mean interaction: 0.06087947303040709
Non-zero entries: 9900


Hamiltonian Construction

1. Hamiltonian A (Polaris Baseline) ------   Interaction/Ising Hamiltonian [H = \sum J_{ij} Z_i Z_j]

2. Hamiltonian B  -------  Diagonal Feature Hamiltonian H = \sum_i h_i Z_i 

3. Hamiltonain C (Graph Hamiltoinian) ----------  Laplacian Hamiltonian (H=D-A)



Hamiltonian A

In [23]:
hamiltonian_A_interaction_ising = {}

for dataset_name, dataset_data in feature_interaction_data.items():
    print(f"Building Hamiltonian A: Interaction/Ising for {dataset_name}...")
    
    J_interaction_matrix = dataset_data["interaction_matrix"]
    J_upper_triangle = np.triu(J_interaction_matrix, k=1)
    
    interaction_ising_terms = []
    
    for feature_i in range(J_upper_triangle.shape[0]):
        for feature_j in range(feature_i + 1, J_upper_triangle.shape[1]):
            coefficient = float(J_upper_triangle[feature_i, feature_j])
            
            if coefficient != 0:
                interaction_ising_terms.append({
                    "coefficient": coefficient,
                    "pauli_string": f"Z{feature_i} Z{feature_j}",
                    "feature_i": feature_i,
                    "feature_j": feature_j
                })
    
    hamiltonian_A_interaction_ising[dataset_name] = {
        "hamiltonian_type": "A_interaction_ising",
        "formula": "H_A = sum_{i<j} J_ij Z_i Z_j",
        "J_interaction_matrix": J_interaction_matrix,
        "selected_features": dataset_data["selected_features"],
        "pauli_terms": interaction_ising_terms,
    }
    
    print(f"{dataset_name}: {len(interaction_ising_terms)} ZZ terms")

Building Hamiltonian A: Interaction/Ising for CYP3A4_Substrate_CarbonMangels...
CYP3A4_Substrate_CarbonMangels: 4950 ZZ terms
Building Hamiltonian A: Interaction/Ising for CYP2D6_Substrate_CarbonMangels...
CYP2D6_Substrate_CarbonMangels: 4950 ZZ terms
Building Hamiltonian A: Interaction/Ising for CYP2C9_Substrate_CarbonMangels...
CYP2C9_Substrate_CarbonMangels: 4950 ZZ terms
Building Hamiltonian A: Interaction/Ising for hERG...
hERG: 4950 ZZ terms
Building Hamiltonian A: Interaction/Ising for AMES...
AMES: 4950 ZZ terms
Building Hamiltonian A: Interaction/Ising for BBB_Martins...
BBB_Martins: 4950 ZZ terms
Building Hamiltonian A: Interaction/Ising for PGP_Broccatelli...
PGP_Broccatelli: 4950 ZZ terms
Building Hamiltonian A: Interaction/Ising for Bioavailability_Ma...
Bioavailability_Ma: 4950 ZZ terms
Building Hamiltonian A: Interaction/Ising for DILI...
DILI: 4851 ZZ terms
Building Hamiltonian A: Interaction/Ising for HIA_Hou...
HIA_Hou: 4950 ZZ terms


Hamiltonian B

In [24]:
hamiltonian_B_diagonal_feature = {}

for dataset_name, dataset_data in selected_feature_data.items():
    print(f"Building Hamiltonian B: Diagonal Feature for {dataset_name}...")
    
    selected_features = dataset_data["selected_features"]
    selected_mi_scores = dataset_data["selected_mi_scores"]
    
    # Hamiltonian B: H_B = sum_i h_i Z_i
    diagonal_feature_terms = []
    
    for local_feature_index, original_feature_index in enumerate(selected_features):
        coefficient = float(selected_mi_scores[local_feature_index])
        
        if coefficient != 0:
            diagonal_feature_terms.append({
                "coefficient": coefficient,
                "pauli_string": f"Z{local_feature_index}",
                "local_feature_index": local_feature_index,
                "original_fingerprint_bit": int(original_feature_index)
            })
    
    hamiltonian_B_diagonal_feature[dataset_name] = {
        "hamiltonian_type": "B_diagonal_feature",
        "formula": "H_B = sum_i h_i Z_i",
        "selected_features": selected_features,
        "selected_mi_scores": selected_mi_scores,
        "X_selected_scaled": dataset_data["X_selected_scaled"],
        "pauli_terms": diagonal_feature_terms
    }
    
    print(f"{dataset_name}: {len(diagonal_feature_terms)} Z terms")

Building Hamiltonian B: Diagonal Feature for CYP3A4_Substrate_CarbonMangels...
CYP3A4_Substrate_CarbonMangels: 100 Z terms
Building Hamiltonian B: Diagonal Feature for CYP2D6_Substrate_CarbonMangels...
CYP2D6_Substrate_CarbonMangels: 100 Z terms
Building Hamiltonian B: Diagonal Feature for CYP2C9_Substrate_CarbonMangels...
CYP2C9_Substrate_CarbonMangels: 100 Z terms
Building Hamiltonian B: Diagonal Feature for hERG...
hERG: 100 Z terms
Building Hamiltonian B: Diagonal Feature for AMES...
AMES: 100 Z terms
Building Hamiltonian B: Diagonal Feature for BBB_Martins...
BBB_Martins: 100 Z terms
Building Hamiltonian B: Diagonal Feature for PGP_Broccatelli...
PGP_Broccatelli: 100 Z terms
Building Hamiltonian B: Diagonal Feature for Bioavailability_Ma...
Bioavailability_Ma: 100 Z terms
Building Hamiltonian B: Diagonal Feature for DILI...
DILI: 100 Z terms
Building Hamiltonian B: Diagonal Feature for HIA_Hou...
HIA_Hou: 100 Z terms


Hamiltonian C

In [25]:
hamiltonian_C_graph_laplacian = {}

for dataset_name, dataset_data in feature_interaction_data.items():
    print(f"Building Hamiltonian C: Graph Laplacian for {dataset_name}...")
    
    adjacency_matrix = dataset_data["interaction_matrix"]

    # Degree matrix: D_ii = sum_j A_ij
    degree_values = np.sum(adjacency_matrix, axis=1)
    degree_matrix = np.diag(degree_values)

    # Hamiltonian C: H_C = D - A
    laplacian_matrix = degree_matrix - adjacency_matrix

    hamiltonian_C_graph_laplacian[dataset_name] = {
        "hamiltonian_type": "C_graph_laplacian",
        "formula": "H_C = D - A",
        "adjacency_matrix": adjacency_matrix,
        "degree_matrix": degree_matrix,
        "laplacian_matrix": laplacian_matrix,
        "selected_features": dataset_data["selected_features"],
        "X_selected_scaled": dataset_data["X_selected_scaled"],
    }

    print(f"{dataset_name}: Laplacian shape = {laplacian_matrix.shape}")

Building Hamiltonian C: Graph Laplacian for CYP3A4_Substrate_CarbonMangels...
CYP3A4_Substrate_CarbonMangels: Laplacian shape = (100, 100)
Building Hamiltonian C: Graph Laplacian for CYP2D6_Substrate_CarbonMangels...
CYP2D6_Substrate_CarbonMangels: Laplacian shape = (100, 100)
Building Hamiltonian C: Graph Laplacian for CYP2C9_Substrate_CarbonMangels...
CYP2C9_Substrate_CarbonMangels: Laplacian shape = (100, 100)
Building Hamiltonian C: Graph Laplacian for hERG...
hERG: Laplacian shape = (100, 100)
Building Hamiltonian C: Graph Laplacian for AMES...
AMES: Laplacian shape = (100, 100)
Building Hamiltonian C: Graph Laplacian for BBB_Martins...
BBB_Martins: Laplacian shape = (100, 100)
Building Hamiltonian C: Graph Laplacian for PGP_Broccatelli...
PGP_Broccatelli: Laplacian shape = (100, 100)
Building Hamiltonian C: Graph Laplacian for Bioavailability_Ma...
Bioavailability_Ma: Laplacian shape = (100, 100)
Building Hamiltonian C: Graph Laplacian for DILI...
DILI: Laplacian shape = (100, 10

----Check----- Hamiltonian Validation check 

In [26]:
print("\n===== HAMILTONIAN VALIDATION =====\n")

for dataset_name in hamiltonian_A_interaction_ising.keys():

    print(f"\n--- {dataset_name} ---")

    # ==========================
    # Hamiltonian A
    # ==========================
    num_A_terms = len(
        hamiltonian_A_interaction_ising[dataset_name]["pauli_terms"]
    )

    print(f"Hamiltonian A (Interaction/Ising):")
    print(f"  ZZ terms = {num_A_terms}")

    # ==========================
    # Hamiltonian B
    # ==========================
    num_B_terms = len(
        hamiltonian_B_diagonal_feature[dataset_name]["pauli_terms"]
    )

    print(f"Hamiltonian B (Diagonal Feature):")
    print(f"  Z terms = {num_B_terms}")

    # ==========================
    # Hamiltonian C
    # ==========================
    laplacian_matrix = (
        hamiltonian_C_graph_laplacian[dataset_name]["laplacian_matrix"]
    )

    is_symmetric = np.allclose(
        laplacian_matrix,
        laplacian_matrix.T
    )

    eigenvalues = np.linalg.eigvalsh(laplacian_matrix)

        # ==========================
    # Maurice Feature Validation
    # ==========================
    x_scaled = hamiltonian_B_diagonal_feature[dataset_name]["X_selected_scaled"]

    print(f"Maurice x_features:")
    print(f"  Shape = {x_scaled.shape}")
    print(f"  Min = {x_scaled.min():.6f}")
    print(f"  Max = {x_scaled.max():.6f}")

    print(f"Hamiltonian C (Graph Laplacian):")
    print(f"  Shape = {laplacian_matrix.shape}")
    print(f"  Symmetric = {is_symmetric}")
    print(f"  Min eigenvalue = {eigenvalues.min():.6f}")
    print(f"  Max eigenvalue = {eigenvalues.max():.6f}")


===== HAMILTONIAN VALIDATION =====


--- CYP3A4_Substrate_CarbonMangels ---
Hamiltonian A (Interaction/Ising):
  ZZ terms = 4950
Hamiltonian B (Diagonal Feature):
  Z terms = 100
Maurice x_features:
  Shape = (670, 100)
  Min = 0.000000
  Max = 1.000000
Hamiltonian C (Graph Laplacian):
  Shape = (100, 100)
  Symmetric = True
  Min eigenvalue = 0.000000
  Max eigenvalue = 5.819001

--- CYP2D6_Substrate_CarbonMangels ---
Hamiltonian A (Interaction/Ising):
  ZZ terms = 4950
Hamiltonian B (Diagonal Feature):
  Z terms = 100
Maurice x_features:
  Shape = (667, 100)
  Min = 0.000000
  Max = 1.000000
Hamiltonian C (Graph Laplacian):
  Shape = (100, 100)
  Symmetric = True
  Min eigenvalue = 0.000000
  Max eigenvalue = 9.957332

--- CYP2C9_Substrate_CarbonMangels ---
Hamiltonian A (Interaction/Ising):
  ZZ terms = 4950
Hamiltonian B (Diagonal Feature):
  Z terms = 100
Maurice x_features:
  Shape = (669, 100)
  Min = 0.000000
  Max = 1.000000
Hamiltonian C (Graph Laplacian):
  Shape = (100, 10

Hamiltonian C padding

In [28]:
hamiltonian_C_graph_laplacian_padded = {}

target_dim = 128

for dataset_name, dataset_data in hamiltonian_C_graph_laplacian.items():
    print(f"Padding Hamiltonian C for {dataset_name}...")
    
    laplacian_matrix = dataset_data["laplacian_matrix"]
    original_dim = laplacian_matrix.shape[0]
    
    padded_laplacian_matrix = np.zeros((target_dim, target_dim))
    padded_laplacian_matrix[:original_dim, :original_dim] = laplacian_matrix
    
    hamiltonian_C_graph_laplacian_padded[dataset_name] = {
        "hamiltonian_type": "C_graph_laplacian_padded",
        "formula": "H_C = D - A, padded from 100x100 to 128x128",
        "original_dim": original_dim,
        "target_dim": target_dim,
        "num_qubits": 7,
        "selected_features": dataset_data["selected_features"],
        "X_selected_scaled": dataset_data["X_selected_scaled"],
        "original_laplacian_matrix": laplacian_matrix,
        "padded_laplacian_matrix": padded_laplacian_matrix,
    }
    
    print(f"{dataset_name}: {original_dim}x{original_dim} → {target_dim}x{target_dim}")

Padding Hamiltonian C for CYP3A4_Substrate_CarbonMangels...
CYP3A4_Substrate_CarbonMangels: 100x100 → 128x128
Padding Hamiltonian C for CYP2D6_Substrate_CarbonMangels...
CYP2D6_Substrate_CarbonMangels: 100x100 → 128x128
Padding Hamiltonian C for CYP2C9_Substrate_CarbonMangels...
CYP2C9_Substrate_CarbonMangels: 100x100 → 128x128
Padding Hamiltonian C for hERG...
hERG: 100x100 → 128x128
Padding Hamiltonian C for AMES...
AMES: 100x100 → 128x128
Padding Hamiltonian C for BBB_Martins...
BBB_Martins: 100x100 → 128x128
Padding Hamiltonian C for PGP_Broccatelli...
PGP_Broccatelli: 100x100 → 128x128
Padding Hamiltonian C for Bioavailability_Ma...
Bioavailability_Ma: 100x100 → 128x128
Padding Hamiltonian C for DILI...
DILI: 100x100 → 128x128
Padding Hamiltonian C for HIA_Hou...
HIA_Hou: 100x100 → 128x128


Validation

In [30]:
for dataset_name, dataset_data in hamiltonian_C_graph_laplacian_padded.items():
    H_padded = dataset_data["padded_laplacian_matrix"]

    print(dataset_name)
    print("Shape:", H_padded.shape)
    print("Symmetric:", np.allclose(H_padded, H_padded.T))
    print("Num qubits:", dataset_data["num_qubits"])
    print("Maurice x_features:", dataset_data["X_selected_scaled"].shape)
    print("-" * 50)

CYP3A4_Substrate_CarbonMangels
Shape: (128, 128)
Symmetric: True
Num qubits: 7
Maurice x_features: (670, 100)
--------------------------------------------------
CYP2D6_Substrate_CarbonMangels
Shape: (128, 128)
Symmetric: True
Num qubits: 7
Maurice x_features: (667, 100)
--------------------------------------------------
CYP2C9_Substrate_CarbonMangels
Shape: (128, 128)
Symmetric: True
Num qubits: 7
Maurice x_features: (669, 100)
--------------------------------------------------
hERG
Shape: (128, 128)
Symmetric: True
Num qubits: 7
Maurice x_features: (655, 100)
--------------------------------------------------
AMES
Shape: (128, 128)
Symmetric: True
Num qubits: 7
Maurice x_features: (7278, 100)
--------------------------------------------------
BBB_Martins
Shape: (128, 128)
Symmetric: True
Num qubits: 7
Maurice x_features: (2030, 100)
--------------------------------------------------
PGP_Broccatelli
Shape: (128, 128)
Symmetric: True
Num qubits: 7
Maurice x_features: (1218, 100)
-------

Hamiltonian C Pauli decomposition 

In [33]:
import itertools
import numpy as np

# Single-qubit Pauli matrices
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

pauli_matrices = {
    "I": I,
    "X": X,
    "Y": Y,
    "Z": Z
}

def kron_n(pauli_string):
    result = pauli_matrices[pauli_string[0]]
    for p in pauli_string[1:]:
        result = np.kron(result, pauli_matrices[p])
    return result

def pauli_decompose_matrix(H, num_qubits=7, tolerance=1e-10):
    dim = 2 ** num_qubits
    pauli_terms = []
    
    for pauli_tuple in itertools.product(["I", "X", "Y", "Z"], repeat=num_qubits):
        pauli_string = "".join(pauli_tuple)
        P = kron_n(pauli_string)
        
        coefficient = np.trace(P.conj().T @ H) / dim
        locality = sum(p != "I" for p in pauli_string)
        
        if abs(coefficient) > tolerance:
            pauli_terms.append({
                "coefficient": float(np.real(coefficient)),
                "pauli_string": pauli_string,
                "locality": locality

            })
    
    return pauli_terms

In [34]:
# Decompose all 10 datasets for Hamiltonian C

hamiltonian_C_graph_laplacian_pauli = {}

for dataset_name, dataset_data in hamiltonian_C_graph_laplacian_padded.items():
    print(f"Pauli decomposing Hamiltonian C for {dataset_name}...")
    
    H_padded = dataset_data["padded_laplacian_matrix"]
    
    pauli_terms = pauli_decompose_matrix(
        H_padded,
        num_qubits=7,
        tolerance=1e-10
    )
    
    hamiltonian_C_graph_laplacian_pauli[dataset_name] = {
        "hamiltonian_type": "C_graph_laplacian_pauli_decomposed",
        "formula": "H_C = sum_k alpha_k P_k",
        "num_qubits": 7,
        "selected_features": dataset_data["selected_features"],
        "X_selected_scaled": dataset_data["X_selected_scaled"],
        "pauli_terms": pauli_terms,
    }
    
    print(f"{dataset_name}: {len(pauli_terms)} Pauli terms")

Pauli decomposing Hamiltonian C for CYP3A4_Substrate_CarbonMangels...
CYP3A4_Substrate_CarbonMangels: 8256 Pauli terms
Pauli decomposing Hamiltonian C for CYP2D6_Substrate_CarbonMangels...
CYP2D6_Substrate_CarbonMangels: 8256 Pauli terms
Pauli decomposing Hamiltonian C for CYP2C9_Substrate_CarbonMangels...
CYP2C9_Substrate_CarbonMangels: 8256 Pauli terms
Pauli decomposing Hamiltonian C for hERG...
hERG: 8256 Pauli terms
Pauli decomposing Hamiltonian C for AMES...
AMES: 8256 Pauli terms
Pauli decomposing Hamiltonian C for BBB_Martins...
BBB_Martins: 8256 Pauli terms
Pauli decomposing Hamiltonian C for PGP_Broccatelli...
PGP_Broccatelli: 8256 Pauli terms
Pauli decomposing Hamiltonian C for Bioavailability_Ma...
Bioavailability_Ma: 8256 Pauli terms
Pauli decomposing Hamiltonian C for DILI...
DILI: 8256 Pauli terms
Pauli decomposing Hamiltonian C for HIA_Hou...
HIA_Hou: 8256 Pauli terms


## Export Curated Hamiltonians

The curated Hamiltonian dictionaries are exported for use by the separate class-based LCU framework.

In [36]:

# ============================================================
# EXPORT CURATED ADMET HAMILTONIANS
# ============================================================

import os
import pickle

export_dir = "curated_admet_hamiltonians"
os.makedirs(export_dir, exist_ok=True)

curated_hamiltonians = {
    "hamiltonian_A_interaction_ising": hamiltonian_A_interaction_ising,
    "hamiltonian_B_diagonal_feature": hamiltonian_B_diagonal_feature,
    "hamiltonian_C_graph_laplacian": hamiltonian_C_graph_laplacian,
    "hamiltonian_C_graph_laplacian_padded": hamiltonian_C_graph_laplacian_padded,
    "hamiltonian_C_graph_laplacian_pauli": hamiltonian_C_graph_laplacian_pauli,
}

export_path = os.path.join(export_dir, "curated_admet_hamiltonians.pkl")

with open(export_path, "wb") as f:
    pickle.dump(curated_hamiltonians, f)

print(f"Saved curated Hamiltonians to: {export_path}")
print("Available keys:")
for key in curated_hamiltonians:
    print("-", key)


Saved curated Hamiltonians to: curated_admet_hamiltonians/curated_admet_hamiltonians.pkl
Available keys:
- hamiltonian_A_interaction_ising
- hamiltonian_B_diagonal_feature
- hamiltonian_C_graph_laplacian
- hamiltonian_C_graph_laplacian_padded
- hamiltonian_C_graph_laplacian_pauli


## Export Curated ADMET Feature Vectors

This section exports molecule-level fingerprint vectors for downstream runtime initialization.

For each dataset, it saves:

- full 1024-bit Morgan fingerprints,
- selected Top-K fingerprint vectors for K = 10, 20, 50, 100,
- selected feature indices,
- SMILES strings,
- ADMET labels.

These exported feature vectors are used as runtime 1D array inputs for quantum state initialization.

In [35]:
# ============================================================
# EXPORT CURATED ADMET FEATURE VECTORS
# ============================================================

import os
import pickle
import numpy as np

top_k_levels = [10, 20, 30, 40, 50, 75, 100]

curated_admet_feature_vectors = {}

for dataset_name, data in selected_feature_data.items():
    print(f"Exporting feature vectors for {dataset_name}...")

    X_full = fingerprint_data[dataset_name]["X"]
    y = fingerprint_data[dataset_name]["y"]
    smiles = fingerprint_data[dataset_name]["smiles"]

    selected_features = data["selected_features"]
    X_selected_all = data["X_selected"]
    X_selected_scaled_all = data["X_selected_scaled"]

    dataset_export = {
        "full_fingerprints": X_full.astype(float),
        "smiles": smiles,
        "labels": y,
        "fingerprint_type": fingerprint_data[dataset_name].get(
            "fingerprint_type",
            "Morgan count fingerprint"
        ),
        "top_k_feature_vectors": {},
    }

    for top_k in top_k_levels:
        if top_k > X_selected_all.shape[1]:
            print(
                f"  Skipping top{top_k}: only "
                f"{X_selected_all.shape[1]} selected features available."
            )
            continue

        dataset_export["top_k_feature_vectors"][f"top{top_k}"] = {
            "fingerprint_vectors": X_selected_all[:, :top_k].astype(float),
            "scaled_feature_vectors": X_selected_scaled_all[:, :top_k].astype(float),
            "selected_feature_indices": selected_features[:top_k].astype(int),
            "num_molecules": int(X_selected_all.shape[0]),
            "num_features": int(top_k),
            "scaling": "minmax_0_1",
        }

        print(
            f"  top{top_k}: "
            f"{X_selected_all[:, :top_k].shape[0]} molecules × {top_k} features"
        )

    curated_admet_feature_vectors[dataset_name] = dataset_export

export_dir = "curated_admet_feature_vectors"
os.makedirs(export_dir, exist_ok=True)

export_path = os.path.join(
    export_dir,
    "curated_admet_feature_vectors.pkl"
)

with open(export_path, "wb") as f:
    pickle.dump(curated_admet_feature_vectors, f)

print("\nSaved curated ADMET feature vectors to:")
print(export_path)

print("\nAvailable datasets:")
for dataset_name in curated_admet_feature_vectors:
    print("-", dataset_name)

print("\nAvailable Top-K levels per dataset:")
for dataset_name, dataset_export in curated_admet_feature_vectors.items():
    print(
        dataset_name,
        list(dataset_export["top_k_feature_vectors"].keys())
    )

Exporting feature vectors for CYP3A4_Substrate_CarbonMangels...
  top10: 670 molecules × 10 features
  top20: 670 molecules × 20 features
  top30: 670 molecules × 30 features
  top40: 670 molecules × 40 features
  top50: 670 molecules × 50 features
  top75: 670 molecules × 75 features
  top100: 670 molecules × 100 features
Exporting feature vectors for CYP2D6_Substrate_CarbonMangels...
  top10: 667 molecules × 10 features
  top20: 667 molecules × 20 features
  top30: 667 molecules × 30 features
  top40: 667 molecules × 40 features
  top50: 667 molecules × 50 features
  top75: 667 molecules × 75 features
  top100: 667 molecules × 100 features
Exporting feature vectors for CYP2C9_Substrate_CarbonMangels...
  top10: 669 molecules × 10 features
  top20: 669 molecules × 20 features
  top30: 669 molecules × 30 features
  top40: 669 molecules × 40 features
  top50: 669 molecules × 50 features
  top75: 669 molecules × 75 features
  top100: 669 molecules × 100 features
Exporting feature vectors